In [ ]:
#https://docs.mistral.ai/capabilities/embeddings/rag_quickstart
from mistralai import Mistral
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv
import requests
import numpy as np
import faiss  #pip intall faiss-cpu
import os
import json
from getpass import getpass

load_dotenv()
api_key= os.getenv("MISTRAL_API_KEY")
client = Mistral(api_key=api_key)

#This downloads the model LOCALLY!!!
model = SentenceTransformer("intfloat/multilingual-e5-base")
#Multilingual e5-base is a transformer based model (so stores context in sentences)
#Based on xlm-roberta-base https://huggingface.co/FacebookAI/xlm-roberta-base

d:\OneDrive\Coding\Python\BeanRAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\OneDrive\Coding\Python\BeanRAG\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\danie\.cache\huggingface\hub\models--intfloat--multilingual-e5-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order 

# Creating Embeddings and Storing them in a vector database

In this file, we first create embeddings from the `structured_chunks.json` file we obtained from `DataProcessing.ipynb` and then store them in a vector database to later use for similarity comparison.

First we load the chunks into python

In [2]:
# Pad naar je JSON-bestand
json_path = "structured_chunks.json"

# JSON-bestand inlezen
with open(json_path, "r", encoding="utf-8") as f:
    chunks = json.load(f)
#data is nu een lijst van dictionaries. Iedere dictionary bevat tenminste de keys "text" en "filename" en mogelijk meer informatie.

In [3]:
print("Aantal chunks:", len(chunks))
print(chunks[0].keys())

Aantal chunks: 12060
dict_keys(['filename', 'hoofdstuk', 'text'])


For the sentence_transformers models we need to create a list of string inputs for the model to take.

In [4]:
chunks_str_list = [chunk.get("text") for chunk in chunks]

Now we obtain the embeddings from the strings

In [ ]:
#we could also choose to normalize the embeddings by setting normalize_embeddings = True
#Then we can use dot product instead of cosine similarity.
embeddings = model.encode_document(chunks_str_list) 
#These embeddings have a dimension of 768

In [11]:
embeddings.shape

(12060, 768)

Great, we have our text embeddings! Now let's store them in a vector database

In [ ]:
#(but first, let's store them in a local .npy file so that we can load if anything goes wrong)
np.save("embeddings_multilingual_e5_base.npy", embeddings)

# Load into Vector Database

Now that we have the embeddings, we want to load them into a vector database.

In [ ]:
d = text_embeddings.shape[1]
BeanRAG_VectorDB = faiss.IndexFlatL2(d)
BeanRAG_VectorDB.add(text_embeddings)

And then we store the vector database to be used in a different script.

In [53]:
print(BeanRAG_VectorDB.ntotal)
faiss.write_index(BeanRAG_VectorDB, "BeanRAG_VectorDB.faiss")
#To read the vector database, use faiss.read_index("BeanRAG_VectorDB.faiss")

50
